# 成员 4：Task 3 有效性检验、回测与交付

**我的任务**：读 3 号交付的完整面板（含技术分 + 基本面分），合成复合得分，
做两分组/五分位/净值回测/Newey-West/分时段检验，输出评级与预期收益分布；并负责最终交付。

**输入**：`数据/step3_panel.csv` + `数据/quarterly_indicators.csv`（3 号产出，已贴进本文件夹）
**输出**：`输出/step4_backtest.csv`（回测绩效）+ `输出/step4_card.csv`（建议卡）
**要点**：回测 T+1 执行、含双边交易成本、一字板不可成交顺延；重叠观测用 Newey-West 修正。

In [1]:
from __future__ import annotations
import io, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from types import SimpleNamespace

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

BASE = os.getcwd()
RAW = os.path.join(BASE, '数据')
OUT = os.path.join(BASE, '输出')
os.makedirs(OUT, exist_ok=True)
STOCK_NAME = '牧原股份'
STOCK_CODE = '002714.SZ'
TRADING_DAYS = 243


In [2]:
def month_end_rule():
    """新版 pandas 用 'ME'，老版本用 'M'"""
    try:
        pd.date_range("2020-01-01", periods=2, freq="ME")
        return "ME"
    except Exception:
        return "M"


class TimingBacktester(object):
    """
    回测口径（写进报告的假设）：
      · 信号在 t 日收盘后才能算出来，所以 t 日收盘建仓、t+1 日起享受收益 -> position = signal.shift(1)
      · 一字涨跌停当天无法成交：开仓/平仓信号遇到一字板顺延到下一个可交易日
      · 交易成本：买入 万2.5，卖出 万2.5 + 印花税 千0.5（按 2023 年后的口径），单边约 0.025%~0.075%
      · 空仓期收益按 0 计（不买货基/国债，属于保守口径）
      · 收益全部用后复权价（含现金红利再投资）
    """

    BUY_COST = 0.00025
    SELL_COST = 0.00075

    def __init__(self, panel: pd.DataFrame, fund: FundamentalFramework):
        self.px = panel
        self.fund = fund
        self.horizons = [1, 5, 10, 20, 60]
        self.trading_days = int(round(243))

    # ---------- 5.1 样本构建 ----------
    def build_sample(self):
        px = self.px
        # 最慢的指标是 MA250 / 250日动量 -> 预热期 250 个交易日
        warm = 250
        px = px.iloc[warm:].copy()
        # 未来收益
        for h in self.horizons:
            px["fwd_ret_%d" % h] = px["close_adj"].shift(-h) / px["close_adj"] - 1.0
        px["fwd_ret_21"] = px["close_adj"].shift(-21) / px["close_adj"] - 1.0     # 约 1 个月
        px["fwd_ret_63"] = px["close_adj"].shift(-63) / px["close_adj"] - 1.0     # 约 3 个月
        # 剔除一字涨跌停（当天无法成交）
        px["tradable"] = (px["is_one_word_board"] == 0).astype(int)
        self.sample = px
        self.note("样本区间 %s ~ %s，共 %d 个交易日（预热期剔除 %d 天，一字板 %d 天标记为不可成交）"
                  % (px.index.min().date(), px.index.max().date(), len(px), warm,
                     int((px["is_one_word_board"] == 1).sum())))
        # 年交易日数按样本自己校准
        yrs = (px.index[-1] - px.index[0]).days / 365.25
        self.trading_days = int(round(len(px) / yrs))
        self.note("样本实测年交易日数 %.0f（用于年化）" % self.trading_days)
        return px

    def note(self, m):
        print("  [Task3] " + m)
        self.log.append(m)

    log = []

    # ---------- 5.2 技术信号分箱 ----------
    def test_signals(self, score_col, freq="daily", bins=2, quintiles=True):
        px = self.sample
        out = []
        for col in [c for c in px.columns if c.startswith("sig_")]:
            s = px[col].where(px["fwd_ret_1"].notna())
            g = px.groupby(px[col]).agg(
                **{"天数": ("ret", "size"),
                   **{"未来%d日收益%%" % h: ("fwd_ret_%d" % h, lambda x: x.mean() * 100)
                      for h in self.horizons}})
            g.index = ["Q1(信号=0)" if i == 0 else "Q2(信号=1)" for i in g.index]
            g["指标"] = col
            out.append(g)
        tab = pd.concat(out) if out else pd.DataFrame()
        if not tab.empty:
            cols = ["指标"] + [c for c in tab.columns if c != "指标"]
            tab = tab[cols].reset_index().rename(columns={"index": "分组"})
            tab.to_csv(os.path.join(OUT, "task3_technical_bins.csv"), index=False, encoding="utf-8-sig")
        # 连续型技术指标的五分位（pd.qcut，进阶项）
        q_rows = []
        for col in ["tech_score", "score_mom_60", "score_ma_gap", "rsi14", "macd_hist"]:
            if col not in px.columns:
                continue
            d = px[[col] + ["fwd_ret_%d" % h for h in self.horizons]].dropna(subset=[col])
            if len(d) < 500:
                continue
            d["五分位"] = pd.qcut(d[col], 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"], duplicates="drop")
            g = d.groupby("五分位", observed=True).agg(
                天数=(col, "size"),
                **{"未来%d日%%" % h: ("fwd_ret_%d" % h, lambda x: x.mean() * 100)
                   for h in self.horizons})
            g.insert(0, "指标", col)
            q_rows.append(g)
        qtab = pd.concat(q_rows) if q_rows else pd.DataFrame()
        if not qtab.empty:
            qtab.reset_index().rename(columns={"五分位": "分组"}).to_csv(
                os.path.join(OUT, "task3_technical_quintiles.csv"), index=False, encoding="utf-8-sig")
        return tab, qtab

    # ---------- 5.3 择时回测（含成本、一字板顺延）----------
    def backtest(self, signal, name, allow_defer=True):
        px = self.sample
        sig = pd.Series(signal, index=px.index).fillna(0).clip(0, 1)
        if allow_defer:
            # 一字板当天不能成交：持仓状态维持前一天的值
            pos = sig.copy()
            locked = px["is_one_word_board"] == 1
            for i in range(1, len(pos)):
                if locked.iloc[i]:
                    pos.iloc[i] = pos.iloc[i - 1]
            sig_exec = pos
        else:
            sig_exec = sig
        position = sig_exec.shift(1).fillna(0)      # t-1 收盘决策，t 日持有
        strat_ret = position * px["ret"]
        turnover = position.diff().abs().fillna(position.abs())
        cost = np.where(position.diff() > 0, self.BUY_COST,
                        np.where(position.diff() < 0, self.SELL_COST, 0.0))
        cost = pd.Series(cost, index=px.index).fillna(0) * turnover
        strat_net = strat_ret - cost
        nav = (1 + strat_net).cumprod()
        bh = (1 + px["ret"]).cumprod()
        m = self.metrics(strat_net, nav, name)
        m["换手次数"] = int((turnover > 0).sum())
        m["持仓天数占比%"] = round(float(position.mean() * 100), 2)
        m["交易成本累计%"] = round(float(cost.sum() * 100), 3)
        m["买入持有年化%"] = round(float((bh.iloc[-1]) ** (self.trading_days / len(px)) - 1) * 100, 2)
        m["买入持有最大回撤%"] = round(float(self.max_drawdown(bh) * 100), 2)
        m["买入持有年化波动%"] = round(float(px["ret"].std() * np.sqrt(self.trading_days) * 100), 2)
        m["买入持有净值"] = round(float(bh.iloc[-1]), 3)
        m["超额年化%"] = round(m["年化收益%"] - m["买入持有年化%"], 2)
        return m, nav, bh, position, strat_net

    def max_drawdown(self, nav):
        roll_max = nav.cummax()
        return float(((nav - roll_max) / roll_max).min())

    def metrics(self, ret, nav, name):
        n = len(ret)
        ann = float(nav.iloc[-1]) ** (self.trading_days / n) - 1 if n > 0 else np.nan
        vol = float(ret.std() * np.sqrt(self.trading_days))
        sharpe = ann / vol if vol and vol > 0 else np.nan
        return {"策略": name,
                "年化收益%": round(ann * 100, 2),
                "年化波动%": round(vol * 100, 2),
                "夏普比率": round(sharpe, 3),
                "最大回撤%": round(self.max_drawdown(nav) * 100, 2),
                "净值终点": round(float(nav.iloc[-1]), 3),
                "胜率%": round(float((ret[ret != 0] > 0).mean() * 100), 2),
                "样本天数": n}

    # ---------- 5.4 基本面月末五分位 ----------
    def fundamental_month_end(self):
        px = self.sample
        rule = month_end_rule()
        me = px.resample(rule).last()
        me["fwd_1m"] = me["close_adj"].shift(-1) / me["close_adj"] - 1.0
        me["fwd_3m"] = me["close_adj"].shift(-3) / me["close_adj"] - 1.0
        me = me[me["fund_score"].notna()]
        cols = {dim: "fund_" + dim for dim in self.fund.dims}
        cols["基本面择时得分"] = "fund_score"
        ind_cols = {"ROE_TTM": "ROE_TTM", "营收单季同比": "营收单季同比", "PE_TTM": "PE_TTM",
                    "PB": "PB", "资产负债率": "资产负债率", "净现比_TTM": "净现比_TTM",
                    "存货周转率_TTM": "存货周转率_TTM"}
        # 季度指标按公告日贴到月末（PIT）
        q = self.fund.ind
        for k in ind_cols:
            if k in q.columns:
                s = pd.Series(q[k].values, index=pd.to_datetime(q["公告日"].values))
                s = s[~s.index.duplicated(keep="last")]
                me[k] = s.reindex(me.index, method="ffill")
        rows = []
        for label, col in list(cols.items()) + list(ind_cols.items()):
            if col not in me.columns:
                continue
            d = me[[col, "fwd_1m", "fwd_3m"]].dropna(subset=[col])
            if len(d) < 30:
                continue
            # expanding 分位：每个月末只用它自己及以前的月末观测排位（题目明确要求）
            try:
                pctl = d[col].expanding(min_periods=12).rank(pct=True) * 100
            except AttributeError:
                pctl = d[col].expanding(min_periods=12).apply(
                    lambda w: (w <= w.iloc[-1]).mean() * 100, raw=False)
            # 估值/负债类指标越小越好，先翻转
            if col in ("PE_TTM", "PB", "资产负债率"):
                pctl = 100 - pctl
            q5 = pd.cut(pctl, bins=[0, 20, 40, 60, 80, 100],
                        labels=["Q1", "Q2", "Q3", "Q4", "Q5"], include_lowest=True)
            d = d.assign(分位=q5).dropna(subset=["分位", "fwd_1m"], how="all")
            g = d.dropna(subset=["fwd_1m"]).groupby("分位", observed=True).agg(
                月末数=("fwd_1m", "size"),
                未来1个月收益均值=("fwd_1m", lambda x: x.mean() * 100),
                未来1个月收益中位数=("fwd_1m", lambda x: x.median() * 100),
                未来3个月收益均值=("fwd_3m", lambda x: x.mean() * 100))
            g.insert(0, "指标", label)
            rows.append(g)
        tab = pd.concat(rows) if rows else pd.DataFrame()
        if not tab.empty:
            tab = tab.reset_index().rename(columns={"分位": "分组"})
            tab.to_csv(os.path.join(OUT, "task3_fundamental_quintiles.csv"),
                       index=False, encoding="utf-8-sig")
        return tab, me

    # ---------- 5.5 组合得分与回测 ----------
    def composite_and_forecast(self, w_tech=0.5, w_fund=0.5):
        px = self.sample
        both = px[px["tech_score"].notna() & px["fund_score"].notna()].copy()
        both["composite"] = w_tech * both["tech_score"] + w_fund * both["fund_score"]
        self.both = both
        rows = []
        try:
            both["复合分位"] = pd.qcut(both["composite"], 5,
                                     labels=["Q1", "Q2", "Q3", "Q4", "Q5"], duplicates="drop")
        except ValueError:
            both["复合分位"] = pd.qcut(both["composite"].rank(method="first"), 5,
                                     labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
        for h in self.horizons + [21, 63]:
            g = both.groupby("复合分位", observed=True)["fwd_ret_%d" % h].mean() * 100
            for k, v in g.items():
                rows.append({"持有期": "%d日" % h, "分组": k, "平均未来收益%": round(float(v), 3),
                             "样本天数": int(both.groupby("复合分位", observed=True)["fwd_ret_%d" % h].size()[k])})
        qt = pd.DataFrame(rows)
        wide = qt.pivot(index="持有期", columns="分组", values="平均未来收益%")
        order = ["%d日" % h for h in self.horizons + [21, 63]]
        wide = wide.reindex([o for o in order if o in wide.index])
        wide["Q5-Q1"] = (wide["Q5"] - wide["Q1"]).round(3)
        wide.round(3).to_csv(os.path.join(OUT, "task3_composite_quintiles.csv"), encoding="utf-8-sig")
        qt.to_csv(os.path.join(OUT, "task3_composite_quintiles_long.csv"),
                  index=False, encoding="utf-8-sig")

        # 多空价差的 Newey-West t 值（加分项：重叠观测的统计推断）
        nw = self.newey_west(both)
        nw.to_csv(os.path.join(OUT, "task3_newey_west.csv"), encoding="utf-8-sig")

        # 分时段稳定性
        sub = self.subperiod(both)
        sub.to_csv(os.path.join(OUT, "task3_subperiod.csv"), encoding="utf-8-sig")

        # ---- 最新得分、评级、预期收益分布 ----
        last = px.iloc[-1]
        latest = {"日期": str(px.index[-1].date()),
                  "收盘价(不复权)": round(float(last["close"]), 2),
                  "后复权价": round(float(last["close_adj"]), 2),
                  "总市值(亿)": round(float(last["mkt_cap"]) / 1e8, 1),
                  "技术面得分": round(float(last["tech_score"]), 1),
                  "基本面择时得分": round(float(last["fund_score"]), 1),
                  "复合得分": round(float(w_tech * last["tech_score"] + w_fund * last["fund_score"]), 1)}
        score = latest["复合得分"]
        if score >= 80:
            rating = "看多 Bullish"
        elif score >= 60:
            rating = "偏多 Moderately Bullish"
        elif score >= 40:
            rating = "中性 Neutral"
        else:
            rating = "谨慎 Cautious"
        latest["评级"] = rating
        # 用「历史同分位区间」的未来收益分布做预测
        bucket = pd.cut([score], bins=[0, 20, 40, 60, 80, 100],
                        labels=["Q1", "Q2", "Q3", "Q4", "Q5"], include_lowest=True)[0]
        hist = both[both["复合分位"] == bucket] if str(bucket) in both["复合分位"].cat.categories else both.iloc[:0]
        for lab, col in [("未来1个月", "fwd_ret_21"), ("未来3个月", "fwd_ret_63")]:
            h = hist[col].dropna()
            latest[lab + "_样本数"] = int(len(h))
            latest[lab + "_均值%"] = round(float(h.mean() * 100), 2) if len(h) else None
            latest[lab + "_中位数%"] = round(float(h.median() * 100), 2) if len(h) else None
            latest[lab + "_25分位%"] = round(float(np.percentile(h, 25) * 100), 2) if len(h) else None
            latest[lab + "_75分位%"] = round(float(np.percentile(h, 75) * 100), 2) if len(h) else len(h) and None
            latest[lab + "_胜率%"] = round(float((h > 0).mean() * 100), 1) if len(h) else None
        latest["所处分位区间"] = str(bucket)
        card = pd.DataFrame([latest]).T.rename(columns={0: "取值"})
        card.index.name = "项目"
        card.to_csv(os.path.join(OUT, "task3_recommendation_card.csv"), encoding="utf-8-sig")
        self.card = card
        self.latest = latest
        return wide, qt, card

    def newey_west(self, both):
        rows = []
        try:
            import statsmodels.api as sm
            for h in self.horizons + [21, 63]:
                d = both[["composite", "fwd_ret_%d" % h]].dropna()
                if len(d) < 200:
                    continue
                y = d["fwd_ret_%d" % h].values
                x = sm.add_constant((d["composite"].values - 50) / 50.0)
                ols = sm.OLS(y, x).fit(cov_type="HAC", cov_kwds={"maxlags": h - 1})
                ols_iid = sm.OLS(y, x).fit()
                rows.append({"持有期": "%d日" % h, "斜率(每+50分)": round(float(ols.params[1]) * 100, 3),
                             "NeweyWest_t": round(float(ols.tvalues[1]), 3),
                             "普通OLS_t": round(float(ols_iid.tvalues[1]), 3),
                             "样本数": len(d)})
        except Exception as e:
            print("  Newey-West 计算失败：%s" % str(e)[:120])
        return pd.DataFrame(rows)

    def subperiod(self, both):
        cuts = [("2015-06-12 前（含杠杆牛）", "2010-01-01", "2015-06-12"),
                ("2015-06-12 ~ 2018 熊市", "2015-06-13", "2018-12-31"),
                ("2019 ~ 2021 非洲猪瘟后涨价周期", "2019-01-01", "2021-12-31"),
                ("2022 ~ 2024 产能过载与亏损", "2022-01-01", "2024-12-31"),
                ("2025 ~ 至今", "2025-01-01", "2030-01-01")]
        rows = []
        for name, a, b in cuts:
            d = both.loc[a:b]
            if len(d) < 60:
                continue
            row = {"子区间": name, "交易日数": len(d)}
            for h in [20, 60]:
                col = "fwd_ret_%d" % h
                x = d.dropna(subset=[col])
                if len(x) < 30:
                    continue
                hi = x[x["composite"] >= 60][col].mean() * 100
                lo = x[x["composite"] < 40][col].mean() * 100
                row["高分组(≥60)_%d日%%" % h] = round(float(hi), 3)
                row["低分组(<40)_%d日%%" % h] = round(float(lo), 3)
                row["价差_%d日%%" % h] = round(float(hi - lo), 3)
            rows.append(row)
        return pd.DataFrame(rows)

    # ---------- 5.7 逐维度择时 + 权重网格（回答「哪个维度择时最有效」）----------
    def strategy_screen(self, threshold=60.0):
        """
        把每个维度得分单独当成择时信号（得分≥60 就持有，否则空仓），
        并且只在「该维度得分已经可用」的窗口内和买入持有比较，保证公平。
        """
        px = self.sample
        rows = []
        cand = (["tech_score", "tech_score_full", "fund_score", "fund_score_ex_val"]
                + ["fund_" + d for d in self.fund.dims])
        for col in cand:
            if col not in px.columns:
                continue
            d = px[px[col].notna()].copy()
            if len(d) < 400:
                continue
            sig = (d[col] >= threshold).astype(int)
            m, nav, bh, pos, ret = self.backtest(sig, col + "≥%d" % threshold)
            # 同窗口买入持有
            bh_nav = (1 + d["ret"]).cumprod()
            m["窗口起点"] = str(d.index.min().date())
            m["窗口天数"] = len(d)
            m["同窗口买入持有年化%"] = round(
                float(bh_nav.iloc[-1] ** (self.trading_days / len(d)) - 1) * 100, 2)
            m["同窗口买入持有夏普"] = round(
                float(d["ret"].mean() / d["ret"].std() * np.sqrt(self.trading_days)), 3)
            m["同窗口买入持有最大回撤%"] = round(float(self.max_drawdown(bh_nav) * 100), 2)
            m["相对同窗口年化超额%"] = round(m["年化收益%"] - m["同窗口买入持有年化%"], 2)
            m["夏普改善"] = round(m["夏普比率"] - m["同窗口买入持有夏普"], 3)
            m["回撤改善pct"] = round(m["最大回撤%"] - m["同窗口买入持有最大回撤%"], 2)
            rows.append(m)
        # 权重网格：技术面 vs 基本面
        both = px[px["tech_score"].notna() & px["fund_score"].notna()]
        for w in [0.0, 0.25, 0.5, 0.75, 1.0]:
            s = w * both["tech_score"] + (1 - w) * both["fund_score"]
            sig = (s >= threshold).astype(int).reindex(px.index).fillna(0)
            m, nav, bh, pos, ret = self.backtest(sig, "复合 w_tech=%.2f" % w)
            d = both
            bh_nav = (1 + d["ret"]).cumprod()
            m["窗口起点"] = str(d.index.min().date())
            m["窗口天数"] = len(d)
            m["同窗口买入持有年化%"] = round(
                float(bh_nav.iloc[-1] ** (self.trading_days / len(d)) - 1) * 100, 2)
            m["同窗口买入持有夏普"] = round(
                float(d["ret"].mean() / d["ret"].std() * np.sqrt(self.trading_days)), 3)
            m["同窗口买入持有最大回撤%"] = round(float(self.max_drawdown(bh_nav) * 100), 2)
            m["相对同窗口年化超额%"] = round(m["年化收益%"] - m["同窗口买入持有年化%"], 2)
            m["夏普改善"] = np.nan
            m["回撤改善pct"] = round(m["最大回撤%"] - m["同窗口买入持有最大回撤%"], 2)
            rows.append(m)
        tab = pd.DataFrame(rows)
        front = ["策略", "年化收益%", "年化波动%", "夏普比率", "最大回撤%", "净值终点",
                 "持仓天数占比%", "换手次数", "交易成本累计%", "窗口起点", "窗口天数",
                 "同窗口买入持有年化%", "同窗口买入持有夏普", "同窗口买入持有最大回撤%",
                 "相对同窗口年化超额%", "回撤改善pct"]
        tab = tab[[c for c in front if c in tab.columns]]
        tab.to_csv(os.path.join(OUT, "task3_strategy_screen.csv"), index=False, encoding="utf-8-sig")
        self.screen = tab
        return tab

    # ---------- 5.8 分位组的时间构成（用来解释为什么某些维度有效）----------
    def quintile_dates(self, me):
        """把基本面月末五分位里 Q1/Q5 的具体月份列出来，避免脱离时间背景解读收益"""
        rows = []
        rule = month_end_rule()
        for label in ["基本面择时得分", "fund_估值水平", "fund_盈利能力", "fund_现金流质量"]:
            col = "fund_score" if label == "基本面择时得分" else label
            if col not in me.columns:
                continue
            d = me[[col, "close_adj"]].dropna()
            try:
                p = d[col].expanding(min_periods=12).rank(pct=True) * 100
            except AttributeError:
                p = d[col].expanding(min_periods=12).apply(
                    lambda w: (w <= w.iloc[-1]).mean() * 100, raw=False)
            for q in ["Q1", "Q5"]:
                lo, hi = {"Q1": (0, 20), "Q5": (80, 100.01)}[q]
                sel = d.index[(p >= lo) & (p < hi)]
                year_dist = pd.Series(sel.year).value_counts().sort_index()
                rows.append({"指标": label, "分位": q, "月数": len(sel),
                             "年份分布": ", ".join("%d年%d个月" % (int(y), int(n))
                                                  for y, n in year_dist.items()),
                             "首月": str(sel.min().date()) if len(sel) else "",
                             "末月": str(sel.max().date()) if len(sel) else ""})
        t = pd.DataFrame(rows)
        t.to_csv(os.path.join(OUT, "task3_quintile_dates.csv"), index=False, encoding="utf-8-sig")
        return t
    def plot_backtest(self, results):
        fig, axes = plt.subplots(2, 1, figsize=(13, 8), gridspec_kw={"height_ratios": [2.2, 1]},
                                 sharex=True)
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
        for i, (name, nav) in enumerate(results.items()):
            axes[0].plot(nav.index, nav.values, lw=1.3, color=colors[i % len(colors)], label=name)
        axes[0].set_yscale("log")
        axes[0].set_title("择时策略净值 vs 买入持有（后复权，含交易成本，对数轴）")
        axes[0].legend(fontsize=9, loc="upper left")
        axes[0].grid(alpha=0.25)
        for i, (name, nav) in enumerate(results.items()):
            dd = nav / nav.cummax() - 1
            axes[1].plot(dd.index, dd.values * 100, lw=1.0, color=colors[i % len(colors)], label=name)
        axes[1].set_ylabel("回撤 %")
        axes[1].legend(fontsize=8, loc="lower left", ncol=3)
        axes[1].grid(alpha=0.25)
        axes[1].xaxis.set_major_locator(mdates.YearLocator())
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        fig.tight_layout()
        p = os.path.join(OUT, "task3_backtest.png")
        fig.savefig(p, dpi=140)
        plt.close(fig)
        print("  [图] 已保存 %s" % p)

    def plot_quintiles(self):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
        panels = [("task3_technical_quintiles.csv", "技术指标五分位 vs 未来收益"),
                  ("task3_fundamental_quintiles.csv", "基本面月末五分位 vs 未来1个月收益"),
                  ("task3_composite_quintiles_long.csv", "复合得分五分位 vs 未来收益")]
        for ax, (f, title) in zip(axes, panels):
            try:
                df = pd.read_csv(os.path.join(OUT, f))
            except Exception:
                ax.set_title(title + "\n(无数据)")
                continue
            if "未来20日%" in df.columns:
                metric, lab = "未来20日%", "%d日"
                df["持有期"] = "20日"
                sel = df[df["持有期"] == "20日"] if "持有期" in df.columns else df
                sel = df[df["指标"].isin(df["指标"].unique()[:4])] if "指标" in df.columns else df
                piv = sel.pivot_table(index="分组", columns="指标", values=metric, observed=True)
            elif "未来1个月收益均值" in df.columns:
                sel = df[df["指标"].isin(df["指标"].unique()[:4])]
                piv = sel.pivot_table(index="分组", columns="指标", values="未来1个月收益均值", observed=True)
            else:
                piv = df.pivot_table(index="分组", columns="持有期", values="平均未来收益%",
                                     observed=True, aggfunc="mean")
                piv = piv[[c for c in ["1日", "5日", "20日", "60日"] if c in piv.columns]]
            if piv.empty:
                ax.set_title(title + "\n(无数据)")
                continue
            piv.plot(kind="bar", ax=ax, width=0.78)
            ax.axhline(0, color="black", lw=0.8)
            ax.set_title(title, fontsize=10)
            ax.set_ylabel("%")
            ax.legend(fontsize=7, ncol=2)
            ax.grid(alpha=0.25, axis="y")
        fig.tight_layout()
        p = os.path.join(OUT, "task3_quintile_returns.png")
        fig.savefig(p, dpi=140)
        plt.close(fig)
        print("  [图] 已保存 %s" % p)


# ====================================================================================

In [3]:
# ---- 读 3 号的完整面板 + 季度指标 ----
panel = pd.read_csv(os.path.join(RAW, 'step3_panel.csv'), parse_dates=['trade_date'])
panel = panel.set_index('trade_date').sort_index()
q = pd.read_csv(os.path.join(RAW, 'quarterly_indicators.csv'))

# 构造轻量基本面对象（TimingBacktester 只用到 dims 与 ind）
fund_lite = SimpleNamespace(
    dims=['盈利能力', '成长能力', '偿债能力', '运营效率', '现金流质量', '估值水平'],
    ind=q)

bt = TimingBacktester(panel, fund_lite)
sample = bt.build_sample()
tech_bins, tech_q5 = bt.test_signals('tech_score')
fund_q, month_end = bt.fundamental_month_end()
wide, qlong, card = bt.composite_and_forecast(0.5, 0.5)

# ---- 三个核心择时策略回测（T+1、含成本、一字板顺延）----
sig_t = (sample['tech_score'] >= 60).astype(int)
sig_f = (sample['fund_score'] >= 60).astype(int)
sig_c = (0.5 * sample['tech_score'] + 0.5 * sample['fund_score'] >= 60).astype(int)
mt, nt, bh, pt, rt = bt.backtest(sig_t, '技术面≥60')
mf, nf, bh, pf, rf = bt.backtest(sig_f, '基本面≥60')
mc, nc, bh, pc, rc = bt.backtest(sig_c, '复合≥60')
perf = pd.DataFrame([mt, mf, mc])
perf.to_csv(os.path.join(OUT, 'step4_backtest.csv'), index=False, encoding='utf-8-sig')
card.T.to_csv(os.path.join(OUT, 'step4_card.csv'), encoding='utf-8-sig')

print('step4 完成：回测绩效如下')
print(perf.to_string(index=False))
print()
print('复合得分五分位 Q5-Q1（%）:')
print(wide[['20日', '60日']].to_string() if '20日' in wide.columns else wide.to_string())
print()
print('投资建议卡：')
print(card.to_string())

  [Task3] 样本区间 2015-02-05 ~ 2026-09-30，共 2807 个交易日（预热期剔除 250 天，一字板 1 天标记为不可成交）
  [Task3] 样本实测年交易日数 241（用于年化）


step4 完成：回测绩效如下
    策略  年化收益%  年化波动%  夏普比率  最大回撤%  净值终点   胜率%  样本天数  换手次数  持仓天数占比%  交易成本累计%  买入持有年化%  买入持有最大回撤%  买入持有年化波动%  买入持有净值  超额年化%
技术面≥60  10.63  34.23 0.311 -56.12 3.243 47.40  2807   112    47.17      5.6    27.35     -63.73      44.19  16.706 -16.72
基本面≥60   7.26  23.00 0.316 -54.15 2.263 48.79  2807    10    36.91      0.5    27.35     -63.73      44.19  16.706 -20.09
 复合≥60  10.90  25.99 0.419 -43.38 3.336 48.27  2807    98    35.70      4.9    27.35     -63.73      44.19  16.706 -16.45

复合得分五分位 Q5-Q1（%）:
分组      Q1     Q2     Q3      Q4     Q5  Q5-Q1
持有期                                           
1日   0.166  0.067  0.006   0.283  0.076 -0.090
5日   0.790  0.398 -0.194   1.178  0.695 -0.095
10日  1.451  0.807 -0.410   2.217  1.562  0.111
20日  2.521  2.226 -0.480   3.845  3.312  0.791
60日  7.954  5.583  2.503  12.566  6.847 -1.107
21日  2.633  2.433 -0.547   4.083  3.394  0.761
63日  8.560  5.780  3.146  12.555  7.362 -1.198

投资建议卡：
                     取值
项目                    